### DEG Filtering

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import os

print("Libraries loaded")

In [ ]:
data_processed = os.path.expanduser("~/Pan-Autoimmune-miRNA-ML/data/processed")

datasets = {
    "Vitiligo": pd.read_csv(f"{data_processed}/GSE65127_vitiligo_expr.csv", index_col=0),
    "SLE": pd.read_csv(f"{data_processed}/GSE318067_SLE_expr.csv", index_col=0),
    "RA": pd.read_csv(f"{data_processed}/GSE93272_RA_expr.csv", index_col=0),
    "T1D": pd.read_csv(f"{data_processed}/GSE55098_T1D_expr.csv", index_col=0)
}

for disease, df in datasets.items():
    print(f"{disease}: {df.shape}")

In [ ]:
## Extract sample metadata to identify case/control labels
## GEO files already downloaded — loading from disk (no re-download)

import GEOparse

data_raw = os.path.expanduser("~/Pan-Autoimmune-miRNA-ML/data/raw")

# Load GSE65127 to extract sample metadata
gse65127 = GEOparse.get_GEO(geo="GSE65127", destdir=data_raw, silent=True)

# Look at one sample's metadata
sample_id = list(gse65127.gsms.keys())[0]
print(sample_id)
print(gse65127.gsms[sample_id].metadata.keys())

In [ ]:
## Inspect title metadata to extract case/control labels
# Print all sample titles to understand case/control labeling
for gsm_id, gsm in gse65127.gsms.items():
    print(gsm_id, gsm.metadata['title'][0])

In [ ]:
## Assign case/control labels for Vitiligo (GSE65127)
vitiligo_labels = {}
for gsm_id, gsm in gse65127.gsms.items():
    title = gsm.metadata['title'][0]
    if 'Healthy Volunteer' in title:
        vitiligo_labels[gsm_id] = 'control'
    elif 'Lesional' in title:  # catches Lesional, Non Lesional, Peri-Lesional
        vitiligo_labels[gsm_id] = 'case'

print(f"Cases: {sum(v=='case' for v in vitiligo_labels.values())}")
print(f"Controls: {sum(v=='control' for v in vitiligo_labels.values())}")

In [ ]:
## Load remaining GEO objects and inspect sample titles
gse_objects = {
    "SLE": GEOparse.get_GEO(geo="GSE318067", destdir=data_raw, silent=True),
    "RA": GEOparse.get_GEO(geo="GSE93272", destdir=data_raw, silent=True),
    "T1D": GEOparse.get_GEO(geo="GSE55098", destdir=data_raw, silent=True)
}

for disease, gse in gse_objects.items():
    print(f"\n--- {disease} ---")
    for gsm_id, gsm in list(gse.gsms.items())[:5]:
        print(gsm_id, gsm.metadata['title'][0])

In [ ]:
## Get unique title patterns for each disease

for disease, gse in gse_objects.items():
    print(f"\n--- {disease} unique patterns ---")
    titles = set()
    for gsm_id, gsm in gse.gsms.items():
        title = gsm.metadata['title'][0]
        print(title)

In [ ]:
## Assign case/control labels for SLE, RA, and T1D based on title inspection above

# SLE (GSE318067)
sle_labels = {}
for gsm_id, gsm in gse_objects["SLE"].gsms.items():
    title = gsm.metadata['title'][0]
    if 'SLE_Control' in title:
        sle_labels[gsm_id] = 'control'
    elif title.startswith('SLE'):
        sle_labels[gsm_id] = 'case'

# RA (GSE93272)
ra_labels = {}
for gsm_id, gsm in gse_objects["RA"].gsms.items():
    title = gsm.metadata['title'][0]
    if 'healthy control' in title:
        ra_labels[gsm_id] = 'control'
    elif 'rheumatoid arthritis' in title:
        ra_labels[gsm_id] = 'case'

# T1D (GSE55098)
t1d_labels = {}
for gsm_id, gsm in gse_objects["T1D"].gsms.items():
    title = gsm.metadata['title'][0]
    if 'Normal control' in title:
        t1d_labels[gsm_id] = 'control'
    elif 'Newly-diagnosed' in title:
        t1d_labels[gsm_id] = 'case'

# Summary
for disease, labels in [("SLE", sle_labels), ("RA", ra_labels), ("T1D", t1d_labels)]:
    cases = sum(v=='case' for v in labels.values())
    controls = sum(v=='control' for v in labels.values())
    print(f"{disease} — Cases: {cases}, Controls: {controls}")

In [ ]:
## Check distribution of p-values and fold changes before filtering
for disease in ["Vitiligo", "SLE", "RA", "T1D"]:
    full = deg_results[disease]["full"]
    sig_raw = full[full['pvalue'] < 0.05]
    sig_fc = full[full['log2FC'].abs() > 1]
    sig_adj = full[full['adj_pvalue'] < 0.05]
    print(f"{disease}: raw p<0.05: {len(sig_raw)}, |log2FC|>1: {len(sig_fc)}, adj_p<0.05: {len(sig_adj)}")

In [ ]:
## DEG analysis using independent t-test + Benjamini-Hochberg correction

def run_deg_analysis(expr_df, labels, disease_name, data_processed):
    
    # Separate case and control samples
    case_samples = [s for s, l in labels.items() if l == 'case' and s in expr_df.columns]
    control_samples = [s for s, l in labels.items() if l == 'control' and s in expr_df.columns]
    
    case_data = expr_df[case_samples]
    control_data = expr_df[control_samples]
    
    # Run t-test for each gene
    results = []
    for gene in expr_df.index:
        case_vals = case_data.loc[gene].values
        ctrl_vals = control_data.loc[gene].values
        t_stat, p_val = stats.ttest_ind(case_vals, ctrl_vals)
        log2fc = case_vals.mean() - ctrl_vals.mean()
        results.append({'gene': gene, 'log2FC': log2fc, 'pvalue': p_val})
    
    results_df = pd.DataFrame(results).set_index('gene')
    
    # BH correction
    _, adj_pvals, _, _ = multipletests(results_df['pvalue'].fillna(1), method='fdr_bh')
    results_df['adj_pvalue'] = adj_pvals
    
    # Filter significant DEGs
    sig_degs = results_df[
        (results_df['adj_pvalue'] < 0.05)
    ]
    
    # Save
    filename = f"{disease_name}_DEGs.csv"
    sig_degs.to_csv(f"{data_processed}/{filename}")
    print(f"{disease_name}: {len(sig_degs)} DEGs — Up: {(sig_degs['log2FC']>0).sum()}, Down: {(sig_degs['log2FC']<0).sum()}")
    
    return results_df, sig_degs

In [ ]:
## Run DEG analysis for all four diseases
deg_results = {}

for disease, expr_df, labels in [
    ("Vitiligo", datasets["Vitiligo"], vitiligo_labels),
    ("SLE", datasets["SLE"], sle_labels),
    ("RA", datasets["RA"], ra_labels),
    ("T1D", datasets["T1D"], t1d_labels)
]:
    full_results, sig_degs = run_deg_analysis(expr_df, labels, disease, data_processed)
    deg_results[disease] = {"full": full_results, "sig": sig_degs}

In [ ]:
## Check for anchor genes PTPN22 and NLRP1 in DEG results
anchor_genes = ['PTPN22', 'NLRP1']

for disease in ["Vitiligo", "SLE", "RA", "T1D"]:
    sig = deg_results[disease]["sig"]
    found = [g for g in anchor_genes if g in sig.index]
    print(f"{disease}: {found if found else 'Not in significant DEGs'}")

In [ ]:
## Check PTPN22 and NLRP1 expression trends across all diseases
for disease in ["Vitiligo", "SLE", "RA", "T1D"]:
    full = deg_results[disease]["full"]
    for gene in anchor_genes:
        if gene in full.index:
            row = full.loc[gene]
            print(f"{disease} - {gene}: log2FC={row['log2FC']:.3f}, adj_p={row['adj_pvalue']:.3f}")

In [ ]:
## Save full DEG results for all diseases
for disease in ["Vitiligo", "SLE", "RA", "T1D"]:
    full = deg_results[disease]["full"]
    full.to_csv(f"{data_processed}/{disease}_full_DEG_results.csv")
    print(f"Saved: {disease}_full_DEG_results.csv")

In [ ]:
## Save filtered DEG gene lists for network construction
for disease in ["Vitiligo", "SLE", "RA", "T1D"]:
    deg_results[disease]["sig"].to_csv(f"{data_processed}/{disease}_DEGs.csv")
    print(f"Saved: {disease}_DEGs.csv — {len(deg_results[disease]['sig'])} genes")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_volcano(disease, results_df, save_path):
    fig, ax = plt.subplots(figsize=(8, 6))
    
    log2fc = results_df['log2FC']
    pval = -np.log10(results_df['adj_pvalue'].replace(0, 1e-300))
    
    colors = ['grey'] * len(results_df)
    for i, (fc, p) in enumerate(zip(log2fc, results_df['adj_pvalue'])):
        if p < 0.05 and fc > 1: colors[i] = 'red'
        elif p < 0.05 and fc < -1: colors[i] = 'blue'
    
    ax.scatter(log2fc, pval, c=colors, alpha=0.5, s=10)
    ax.axhline(-np.log10(0.05), color='black', linestyle='--', linewidth=0.8)
    ax.axvline(1, color='black', linestyle='--', linewidth=0.8)
    ax.axvline(-1, color='black', linestyle='--', linewidth=0.8)
    ax.set_xlabel('log2 Fold Change')
    ax.set_ylabel('-log10 Adjusted P-value')
    ax.set_title(f'Volcano Plot — {disease}')
    
    plt.tight_layout()
    plt.savefig(f"{save_path}/{disease}_volcano.png", dpi=300)
    plt.show()
    print(f"Saved: {disease}_volcano.png")

figures_path = os.path.expanduser("~/Pan-Autoimmune-miRNA-ML/results/figures")
os.makedirs(figures_path, exist_ok=True)

for disease in ["Vitiligo", "SLE", "RA", "T1D"]:
    full = deg_results[disease]["full"]
    plot_volcano(disease, full, figures_path)